# Cambrian-P: ask questions about your own videos

Runs [Cambrian-P](https://github.com/cambrian-mllm/cambrian-p) on one video or a whole folder of videos in your Google Drive. For each video it answers your questions and plots the camera path the model predicts.

**Before you start:** Runtime → Change runtime type → **A100 GPU**. The 7B model needs about 15 GB for its weights alone, so the free T4 is too small. An L4 (24 GB) may work with fewer frames (set `NUM_FRAMES` to 32).

**How to use**
1. Put your videos in one Drive folder and set `VIDEO_PATH` to that folder (or to a single video file). Fill in the questions in the settings cell and run it.
2. Run the rest of the cells in order (Runtime → Run all works too).
3. The first time takes about 20–30 minutes (install + 15 GB model download). Run the optional backup cell once afterwards, and later sessions restore from Drive in a few minutes.

The model loads once and then processes every video in the folder. Within one session you can change the videos or questions and rerun only the settings cell and the last three cells.

**What you get**, in `OUTPUT_DIR` on your Drive:
- `results.csv`: one row per question, with the true answer you gave and the model's answer (opens in Google Sheets).
- One folder per video, containing:
  - `frames.jpg`: some of the frames the model saw. Check that they are upright.
  - `trajectory.png`: the predicted camera path, from above and in 3D.
  - `poses.npz` and `trajectory_tum.txt`: the raw per-frame camera poses.

**Recording tips:** hold the phone in landscape. Portrait clips can reach the model sideways, which you would see in `frames.jpg`. If a video fails to load, it is usually HEVC: set the iPhone camera to "Most Compatible" (Settings → Camera → Formats) or convert the clip to H.264 mp4.

**Camera poses:** every checkpoint predicts them. The paper's pose results use `Cambrian-P-7B-Mix-MA` (doc/evaluation.md), so try it when the path matters most. The path starts at the first frame. Distances are in the model's own relative units, not meters, so compare shapes (does a loop close, is a turn 90°) rather than lengths.

**Check the load worked:** the run log should contain a `token_num:` line, `[cam-load] ✓ Loaded model.rec_head.camera_tokens`, and `SUCCESS: All weights in new_state_dict match the loaded model.` Without them, the camera tokens are not being used.

In [ ]:
# @title 1. Settings
VIDEO_PATH = "/content/drive/MyDrive/videos"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/cambrianp_results"  # @param {type:"string"}
MODEL = "Cambrian-P-7B"  # @param ["Cambrian-P-7B", "Cambrian-P-7B-32f", "Cambrian-P-7B-Mix-MA", "Cambrian-P-7B-Mix-3R", "Cambrian-P-7B-Mix-CamS"]
NUM_FRAMES = 0  # @param {type:"integer"}
BACKUP_PATH = "/content/drive/MyDrive/cambrianp_env.tar"  # @param {type:"string"}
BACKUP_INCLUDES_MODEL = False  # @param {type:"boolean"}
REPO_URL = "https://github.com/cambrian-mllm/cambrian-p.git"  # @param {type:"string"}

# VIDEO_PATH is a single video or a folder; every video in the folder is processed.

# These two helpers phrase a question the way VSI-Bench does (lmms-eval/lmms_eval/tasks/vsibench/utils.py),
# which is the format the model was trained and scored on.
def number_q(question):
    """For a number or a single word: counts, distances (m), object sizes (cm), room area (m²)."""
    return f"These are frames of a video.\n{question}\nPlease answer the question using a single word or phrase."

def choice_q(question, options):
    """Multiple choice: relative direction or distance, route planning, appearance order."""
    lettered = "\n".join(f"{letter}. {option}" for letter, option in zip("ABCDEFGH", options))
    return f"These are frames of a video.\n{question}\nOptions:\n{lettered}\nAnswer with the option's letter from the given choices directly."

# Asked on every video. Each entry is a question, or [question, true answer] to compare against.
QUESTIONS = [
    "Describe the path the camera takes through the scene.",
    number_q("How many chairs are in this room?"),
]

# Extra questions for particular videos, keyed by file name. Leave it as {} if you have none.
PER_VIDEO_QUESTIONS = {
    "living_room.mp4": [
        [number_q("What is the distance between the sofa and the TV (in meters)?"), "2.4"],
        [choice_q("If I am standing by the sofa and facing the TV, is the window to my left, right, or back?",
                  ["left", "right", "back"]), "A"],
    ],
}

# NUM_FRAMES = 0 uses the frame count each model was evaluated with (doc/evaluation.md).
DEFAULT_FRAMES = {
    "Cambrian-P-7B": 128, "Cambrian-P-7B-Mix-CamS": 128,
    "Cambrian-P-7B-Mix-MA": 64,
    "Cambrian-P-7B-32f": 32, "Cambrian-P-7B-Mix-3R": 32,
}
frames = NUM_FRAMES or DEFAULT_FRAMES[MODEL]

import json, os
os.environ.update({
    "VIDEO_PATH": VIDEO_PATH, "OUTPUT_DIR": OUTPUT_DIR, "MODEL": MODEL, "FRAMES": str(frames),
    "BACKUP_PATH": BACKUP_PATH, "BACKUP_INCLUDES_MODEL": str(int(BACKUP_INCLUDES_MODEL)),
    "REPO_URL": REPO_URL,
    # Keep uv's Python inside /content so the Drive backup contains everything the venv needs.
    "UV_PYTHON_INSTALL_DIR": "/content/python",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
})
with open("/content/questions.json", "w") as f:
    json.dump({"all": QUESTIONS, "per_video": PER_VIDEO_QUESTIONS}, f)
print(f"{MODEL}, {frames} frames, {len(QUESTIONS)} question(s) for every video, "
      f"extra questions for {len(PER_VIDEO_QUESTIONS)} video(s)")

In [ ]:
# @title 2. Check the GPU and mount Google Drive
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Runtime → Change runtime type → A100 GPU, then run again.")
print("GPU:", gpu)
if int(gpu.split(",")[-1].split()[0]) < 30000:
    print("Warning: under 30 GB of GPU memory. If you run out of memory, set NUM_FRAMES to 32 in cell 1.")

from google.colab import drive
drive.mount("/content/drive")
VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")
if os.path.isdir(VIDEO_PATH):
    found = sorted(f for f in os.listdir(VIDEO_PATH) if f.lower().endswith(VIDEO_EXTS))
    print(f"{len(found)} video(s) in {VIDEO_PATH}:", ", ".join(found) or "none")
    for name in sorted(set(PER_VIDEO_QUESTIONS) - set(found)):
        print(f"Warning: PER_VIDEO_QUESTIONS names {name!r}, which is not in the folder (names must match exactly).")
elif os.path.isfile(VIDEO_PATH):
    print("Video:", VIDEO_PATH)
else:
    print(f"Warning: nothing found at {VIDEO_PATH}. Check the path in cell 1 (browse it in the Files panel on the left).")

In [ ]:
%%bash
# @title 3. Set up the environment (skips itself if already done)
set -e
cd /content
if /content/venv/bin/python -c "import torch" 2>/dev/null && [ -d /content/cambrian-p ]; then
    echo "Environment already set up in this session."
elif [ -f "$BACKUP_PATH" ]; then
    echo "Restoring environment from $BACKUP_PATH ..."
    tar xf "$BACKUP_PATH" -C /content
    echo "Restored."
else
    echo "Fresh install (about 10-15 minutes) ..."
    [ -d cambrian-p ] || git clone -q "$REPO_URL" cambrian-p
    pip install -q uv
    # The repo is pinned to Python 3.11, which may differ from Colab's own Python.
    uv venv --seed --python 3.11 /content/venv
    source /content/venv/bin/activate
    cd cambrian-p
    # Same steps and order as the README's "Environment Preparation".
    pip install -q -e ./vggt
    pip install -q hydra-core tensorboard iopath wcmatch fvcore
    pip install -q --upgrade pip
    pip install -q -e ".[train]"
    pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
        --index-url https://download.pytorch.org/whl/cu121
    pip install -q accelerate==0.29.3 easydict matplotlib roma evo imageio OpenEXR
    # Used by lmms-eval's model wrapper. flash-attn is skipped: the wrapper always uses PyTorch's
    # built-in attention (sdpa) on torch >= 2.1.2, so it would never be used.
    pip install -q loguru sqlitedict sacrebleu evaluate tenacity pytz
    echo "Install finished."
fi
# pyarrow: newer releases need NumPy 2, but the repo pins NumPy 1.26.1. Also fixes backups made before this pin.
/content/venv/bin/pip install -q pyarrow==15.0.2


In [ ]:
%%bash
# @title 4. Download the model (skips itself if already present)
set -e
CKPT=/content/cambrian-p/ckpts/$MODEL
if [ -f "$CKPT/config.json" ]; then
    echo "$MODEL already downloaded."
else
    echo "Downloading $MODEL (about 15 GB) ..."
    /content/venv/bin/huggingface-cli download "nyu-visionx/$MODEL" --local-dir "$CKPT"
fi

In [ ]:
%%bash
# @title 5. (Optional, run once) Save the environment to Drive for faster future sessions
# Without the model the archive is several GB; with it, add about 15 GB (needs paid Drive storage).
set -e
cd /content
EXCLUDE="--exclude=cambrian-p/ckpts"
[ "$BACKUP_INCLUDES_MODEL" = "1" ] && EXCLUDE=""
echo "Writing $BACKUP_PATH ..."
tar cf "$BACKUP_PATH" $EXCLUDE python venv cambrian-p
ls -lh "$BACKUP_PATH"

In [ ]:
%%writefile /content/run_videos.py
"""Answer questions about one or more videos with Cambrian-P and plot each video's predicted camera path.

The model is loaded once. For each video it writes, under OUTPUT_DIR/<video name>/:
  answers.json        questions, true answers (if given) and the model's answers
  frames.jpg          a few of the frames the model saw (check the orientation here)
  trajectory.png      bird's-eye and 3D view of the predicted camera path
  poses.npz           per-frame camera-to-world matrices, focal lengths, timestamps
  trajectory_tum.txt  the same path in TUM format, for tools like `evo`
and OUTPUT_DIR/results.csv with one row per question across all videos.
"""
import argparse
import csv
import glob
import json
import os
import traceback

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from decord import VideoReader, cpu
from PIL import Image
from scipy.spatial.transform import Rotation

import cambrianp.utils as cambrianp_utils
from lmms_eval.models.simple.cambrian_p import CambrianP, expand2square
from cambrianp.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from cambrianp.conversation import conv_templates
from cambrianp.mm_utils import tokenizer_image_token
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

VIDEO_EXTS = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")

ap = argparse.ArgumentParser()
ap.add_argument("--model_path", required=True)
ap.add_argument("--videos", required=True, help="a video file, or a folder of videos")
ap.add_argument("--questions_file", required=True)
ap.add_argument("--output_dir", required=True)
ap.add_argument("--num_frames", type=int, default=128)
args = ap.parse_args()

if os.path.isdir(args.videos):
    videos = sorted(p for p in glob.glob(os.path.join(args.videos, "*")) if p.lower().endswith(VIDEO_EXTS))
else:
    videos = [args.videos]
if not videos:
    raise SystemExit(f"No videos found at {args.videos}")

# {"all": [...], "per_video": {"name.mp4": [...]}}; each entry is a question or [question, true answer].
with open(args.questions_file) as f:
    question_spec = json.load(f)


def questions_for(video):
    entries = question_spec.get("all", []) + question_spec.get("per_video", {}).get(os.path.basename(video), [])
    return [(e, "") if isinstance(e, str) else (e[0], str(e[1])) for e in entries]


# For any checkpoint path containing "cambrian", the loader converts the config to LLaVA-OneVision
# style (cambrianp/utils.py). That conversion is meant for starting training from Cambrian-S. Released
# Cambrian-P checkpoints were saved already converted, so converting again overwrites vocab_size
# (152064 vs the weights' 151672) and deletes the camera-head settings, which silently drops the
# per-frame camera tokens. Keep the checkpoint's config as saved, and switch the camera head on
# (use_camera_tokens only takes effect with load_rec_model, llava_arch.py:556; the pose eval sets it too).
def _keep_checkpoint_config(config, model_path):
    config.load_rec_model = True
    if not os.path.isfile(str(getattr(config, "rec_model_path", None))):
        # Path to the authors' VGGT init file; the trained head weights come from the checkpoint itself.
        config.rec_model_path = "None"
    return config

cambrianp_utils.cast_cambrian_config_to_llava_ov_style = _keep_checkpoint_config

# Flags from doc/evaluation.md.
m = CambrianP(pretrained=args.model_path, model_name="llava_qwen", conv_template="qwen_1_5",
              max_frames_num=args.num_frames, use_camera_tokens=True, camera_tokens_mode="camera_tokens",
              camera_tokens_place="append_to_frame", query_mode="query_after_image")
pad_id = m.tokenizer.pad_token_id if m.tokenizer.pad_token_id is not None else m.tokenizer.eos_token_id


def ask(pixels, image_size, video, question, gen_kwargs=None):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), m.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).to(m.device)
    return m._run_inference_step(ids, ids.ne(pad_id), [pixels], image_size, "video", dict(gen_kwargs or {}), video)[0]


def decode_poses(preds):
    """Camera-to-world 4x4 matrices and focal lengths, decoded the same way as extract_pose_llava in
    cambrianp/eval/relpose/unified_pose_eval.py. Frame 0 is the origin; OpenCV axes (x right, y down,
    z forward). Translation is in the model's normalized units, not meters."""
    extrinsic, intrinsic = pose_encoding_to_extri_intri(preds["pose_enc"].float(), (384, 384))
    extrinsic, intrinsic = extrinsic[0].cpu().numpy(), intrinsic[0].cpu().numpy()
    c2w = np.tile(np.eye(4), (len(extrinsic), 1, 1))
    R, t = extrinsic[:, :3, :3], extrinsic[:, :3, 3]
    c2w[:, :3, :3] = R.transpose(0, 2, 1)
    c2w[:, :3, 3] = -np.einsum("nji,nj->ni", R, t)
    return c2w, intrinsic[:, 0, 0]


def plot_trajectory(c2w, times, title, path):
    pos = c2w[:, :3, 3]
    fwd = c2w[:, :3, 2]  # camera viewing direction in world coordinates
    fig = plt.figure(figsize=(13, 6))

    # Bird's-eye view: x (right) against z (forward) of the first frame, for a camera held level.
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(pos[:, 0], pos[:, 2], color="0.75", lw=1, zorder=1)
    sc = ax.scatter(pos[:, 0], pos[:, 2], c=times, cmap="viridis", s=14, zorder=2)
    step = max(1, len(pos) // 16)
    span = max(np.ptp(pos[:, 0]), np.ptp(pos[:, 2]), 1e-6)
    ax.quiver(pos[::step, 0], pos[::step, 2], fwd[::step, 0], fwd[::step, 2], color="tab:orange",
              angles="xy", scale_units="xy", scale=1 / (0.08 * span), width=0.004, zorder=3)
    ax.scatter(*pos[0, [0, 2]], marker="o", s=120, facecolors="none", edgecolors="green", lw=2, label="start", zorder=4)
    ax.scatter(*pos[-1, [0, 2]], marker="X", s=120, color="red", label="end", zorder=4)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("x: right of the first frame")
    ax.set_ylabel("z: forward from the first frame")
    ax.set_title("Bird's-eye view (arrows = where the camera points)")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
    fig.colorbar(sc, ax=ax, label="time in video (s)")

    # 3D view; y is negated so "up" points up.
    ax3 = fig.add_subplot(1, 2, 2, projection="3d")
    ax3.plot(pos[:, 0], pos[:, 2], -pos[:, 1], color="0.6", lw=1)
    ax3.scatter(pos[:, 0], pos[:, 2], -pos[:, 1], c=times, cmap="viridis", s=10)
    ax3.scatter(*pos[0, [0, 2]], -pos[0, 1], color="green", s=60)
    ax3.scatter(*pos[-1, [0, 2]], -pos[-1, 1], color="red", marker="X", s=60)
    ax3.set_xlabel("x (right)")
    ax3.set_ylabel("z (forward)")
    ax3.set_zlabel("up")
    # True proportions, so small up/down jitter is not stretched to fill the box.
    ax3.set_box_aspect(np.maximum(np.ptp(pos[:, [0, 2, 1]], axis=0), 0.05 * span))
    ax3.set_title("3D view")

    fig.suptitle(f"{title}  (units are relative, not meters)")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


def save_frames_preview(frames, path, count=8):
    picks = np.linspace(0, len(frames) - 1, min(count, len(frames)), dtype=int)
    thumbs = [Image.fromarray(frames[i]) for i in picks]
    for t in thumbs:
        t.thumbnail((320, 320))
    w, h = max(t.width for t in thumbs), max(t.height for t in thumbs)
    cols = 4
    sheet = Image.new("RGB", (cols * w, -(-len(thumbs) // cols) * h), "white")
    for k, t in enumerate(thumbs):
        sheet.paste(t, ((k % cols) * w, (k // cols) * h))
    sheet.save(path, quality=85)


def run_video(video, out_dir):
    raw = m._load_video_frames(video, m.max_frames_num)
    # Timestamps of the frames load_video samples (the same uniform spacing over the whole clip).
    vr = VideoReader(video, ctx=cpu(0))
    frame_idx = np.linspace(0, len(vr) - 1, len(raw), dtype=int)
    times = frame_idx / vr.get_avg_fps()
    del vr
    print(f"{len(raw)} frames of {raw.shape[2]}x{raw.shape[1]} over {times[-1]:.1f}s", flush=True)
    save_frames_preview(raw, os.path.join(out_dir, "frames.jpg"))

    # Same frame preprocessing as CambrianP.generate_until for video inputs.
    image_size = list(raw.shape[1:3])
    bg = tuple(int(x * 255) for x in m._image_processor.image_mean)
    frames = [expand2square(Image.fromarray(f, mode="RGB"), bg) for f in raw]
    pixels = m._image_processor.preprocess(frames, return_tensors="pt")["pixel_values"].half().cuda()

    # The model computes camera poses once per generate() and then caches them on itself
    # (llava_qwen.py, "rec_head_preds"), so clear the previous video's poses first.
    m.model.rec_head_preds = None
    rows = []
    questions = questions_for(video)
    for question, truth in questions:
        answer = ask(pixels, image_size, video, question)
        print(f"\nQ: {question}\nA: {answer}" + (f"\n   (true answer: {truth})" if truth else ""), flush=True)
        rows.append({"video": os.path.basename(video), "question": question, "true_answer": truth, "model_answer": answer})
    if not questions:
        # No questions for this video: run one short pass just to get the poses.
        ask(pixels, image_size, video, "Describe this scene.", {"max_new_tokens": 1})
        rows.append({"video": os.path.basename(video), "question": "", "true_answer": "", "model_answer": "(no questions; poses only)"})
    with open(os.path.join(out_dir, "answers.json"), "w") as f:
        json.dump(rows, f, indent=2)

    # The camera tokens sit with the frames, before the question, so the poses do not depend on
    # which question was asked.
    preds = m.model.rec_head_preds
    if preds is None or "pose_enc" not in preds:
        print("Warning: the model returned no camera poses for this video (see the log above).")
        return rows
    c2w, focal = decode_poses(preds)
    np.savez(os.path.join(out_dir, "poses.npz"), c2w=c2w, focal=focal, frame_idx=frame_idx, times=times)
    quat = Rotation.from_matrix(c2w[:, :3, :3]).as_quat()  # x, y, z, w
    np.savetxt(os.path.join(out_dir, "trajectory_tum.txt"), np.column_stack([times, c2w[:, :3, 3], quat]), fmt="%.6f",
               header="timestamp tx ty tz qx qy qz qw")
    plot_trajectory(c2w, times, os.path.basename(video), os.path.join(out_dir, "trajectory.png"))
    print(f"Saved trajectory plot to {out_dir}/trajectory.png")
    return rows


os.makedirs(args.output_dir, exist_ok=True)
all_rows = []
for n, video in enumerate(videos, 1):
    name = os.path.splitext(os.path.basename(video))[0]
    print(f"\n===== [{n}/{len(videos)}] {os.path.basename(video)} =====", flush=True)
    out_dir = os.path.join(args.output_dir, name)
    os.makedirs(out_dir, exist_ok=True)
    try:
        all_rows += run_video(video, out_dir)
    except Exception:
        # Keep going with the other videos; the error is also recorded in results.csv.
        traceback.print_exc()
        all_rows.append({"video": os.path.basename(video), "question": "", "true_answer": "",
                         "model_answer": "ERROR: " + traceback.format_exc().strip().splitlines()[-1]})
    torch.cuda.empty_cache()

with open(os.path.join(args.output_dir, "results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["video", "question", "true_answer", "model_answer"])
    writer.writeheader()
    writer.writerows(all_rows)
print(f"\nDone: {len(videos)} video(s). Answers for all of them are in {args.output_dir}/results.csv")

In [ ]:
%%bash
# @title 6. Run Cambrian-P on your videos
source /content/venv/bin/activate
cd /content/cambrian-p
export PYTHONPATH=$PWD/lmms-eval:$PWD
python /content/run_videos.py \
    --model_path "ckpts/$MODEL" \
    --videos "$VIDEO_PATH" \
    --questions_file /content/questions.json \
    --output_dir "$OUTPUT_DIR" \
    --num_frames "$FRAMES"

In [ ]:
# @title 7. Show the results
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_colwidth", None)
results = pd.read_csv(os.path.join(OUTPUT_DIR, "results.csv"), keep_default_na=False)
for video, rows in results.groupby("video", sort=False):
    display(Markdown(f"## {video}"))
    folder = os.path.join(OUTPUT_DIR, os.path.splitext(video)[0])
    for picture in ("frames.jpg", "trajectory.png"):
        path = os.path.join(folder, picture)
        if os.path.exists(path):
            display(Image(path, width=1000))
    display(rows.drop(columns="video").reset_index(drop=True))